In [ ]:
%load_ext autoreload
%autoreload 2

# Célula 1: Configuração do ambiente e importação da nossa arquitetura
import sys
import os

# Garantir que o notebook enxerga a pasta 'src' na raiz do projeto
sys.path.append(os.path.abspath('..'))

import torch
import numpy as np

# Importar as nossas classes do PEDS
from src.objects import CSstruct, NNstruct, ALstruct
from src.models.surrogate import PEDSModel

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

# Testar a instanciação do motor
cs = CSstruct()
nn_params = NNstruct()
modelo = PEDSModel(nn_params)

print("Arquitetura PEDS carregada com sucesso!")

In [ ]:
import torch, numpy as np
from src.physics.geometry import SimulationDomain
from src.physics.fdfd import FDFDSolver
from src.physics.differentiable_physics import DifferentiableMaxwell
sd = SimulationDomain(Lx=0.95, Ly=17.0, omega=2*np.pi, dpml=2.0,
                        resolution=10, source=1.0, monitor=16.0)
mx = DifferentiableMaxwell(FDFDSolver(sd))
eps = torch.ones(1, sd.ny, sd.nx, requires_grad=True)
rp, ip = mx(eps); (rp.sum() + ip.sum()).backward()
print(bool(torch.isfinite(eps.grad).all()) and float(eps.grad.abs().sum()) > 0)

In [ ]:
# Célula 2: Pipeline de Ingestão de Dados Originais
import os
import torch
import pandas as pd
import numpy as np

# Mapear o caminho da pasta 'data' na raiz do projeto
data_dir = os.path.abspath(os.path.join('..', 'data'))
x_path = os.path.join(data_dir, "X_maxwell10_small.csv")
y_path = os.path.join(data_dir, "y_maxwell10_small.csv")

print("[ETL] A iniciar a ingestão dos ficheiros CSV...")

# 1. Carregar e transformar a Matriz X (Features)
# O Julia guardou como (Features, Samples). O PyTorch exige (Samples, Features).
X_df = pd.read_csv(x_path, header=None)
X_np = X_df.values.T  # Transposição arquitetural da matriz
X_tensor = torch.tensor(X_np, dtype=torch.float32)

# 2. Carregar e tratar o Vetor Y (Respostas Complexas)
def parse_julia_complex(val):
    if pd.isna(val): return 0j
    # Remove espaços vazios e substitui a sintaxe 'im' do Julia pelo 'j' do Python
    return complex(str(val).replace(' ', '').replace('im', 'j'))

y_df = pd.read_csv(y_path, header=None)
y_flat = y_df.values.flatten() # Garantir que é um vetor 1D
y_np = np.array([parse_julia_complex(v) for v in y_flat])
y_tensor = torch.tensor(y_np, dtype=torch.complex64)

print(f"\n[ETL] Ingestão Concluída!")
print(f"Dataset X Global: {X_tensor.shape}")
print(f"Dataset y Global: {y_tensor.shape}")

# 3. Particionamento Exato dos Dados (Conforme o example_maxwell.jl)
# Nota: O Julia usa índices a começar em 1. O Python começa em 0.
# Xv = X[:, 1:1024] -> Python: [:1024]
X_valid = X_tensor[:1024]
y_valid = y_tensor[:1024]

# Xtest = X[:, end-1023:end] -> Python: [-1024:]
X_test = X_tensor[-1024:]
y_test = y_tensor[-1024:]

# Xt = X[:, 1025:end] -> Python: [1024:]
X_train = X_tensor[1024:]
y_train = y_tensor[1024:]

print("\n[Particionamento] Divisão para o ciclo de Active Learning:")
print(f" - Treino (Xt):     {X_train.shape}")
print(f" - Validação (Xv):  {X_valid.shape}")
print(f" - Teste (Xtest):   {X_test.shape}")

In [ ]:
# Célula 4: Treino do Modelo Baseline (Sem Física)
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

# Importar as nossas estruturas
from src.objects import NNstruct
from src.models.surrogate import BaselineModel
from src.models.losses import calculate_nll_loss

print("--- A iniciar o Treino do Modelo Baseline ---")

# 1. Definir a Métrica de Erro Fracionário (Fractional Error - FE)
# Equivalente à função dFE() do Julia
def calculate_fe(rp, ip, y_true):
    y_pred = torch.complex(rp, ip)
    # FE = norm(y_pred - y_true) / norm(y_true)
    return torch.linalg.norm(y_pred - y_true) / torch.linalg.norm(y_true)

# 2. Configurar o Hardware e Subconjunto de Treino (como no artigo: 1280 amostras)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_init = 1280 
batch_size = 64

train_dataset = TensorDataset(X_train[:n_init], y_train[:n_init])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# 3. Inicializar o Modelo e Otimizador
nn_params = NNstruct()
baseline = BaselineModel(nn_params).to(device)
optimizer = Adam(baseline.parameters(), lr=1e-3)

# 4. Loop de Treino (Equivalente ao Flux.@epochs do Julia)
epochs = 10
baseline.train()

for epoch in range(epochs):
    epoch_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        
        # Forward Pass do Baseline (mgen -> pred / mvar)
        z = baseline.mgen(X_batch)
        pred_out = baseline.pred(z) # Devolve (Real, Imaginário)
        vp_out = baseline.mvar(z)   # Devolve (Variância)
        
        # Extrair componentes
        rp = pred_out[:, 0]
        ip = pred_out[:, 1]
        vp = torch.abs(vp_out.squeeze()) + 1e-6 # Garantir variância estritamente positiva
        
        # Calcular Perda e Retropropagar
        loss = calculate_nll_loss((rp, ip, vp), y_batch)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Época {epoch+1}/{epochs} | Loss NLL: {epoch_loss/len(train_loader):.4f}")

# 5. Avaliação no Conjunto de Teste (Cálculo do FE)
baseline.eval()
with torch.no_grad():
    X_test_gpu = X_test.to(device)
    y_test_gpu = y_test.to(device)
    
    z_test = baseline.mgen(X_test_gpu)
    pred_test = baseline.pred(z_test)
    
    rp_test = pred_test[:, 0]
    ip_test = pred_test[:, 1]
    
    fe_score = calculate_fe(rp_test, ip_test, y_test_gpu)

print("\n========================================")
print(f"Resultado Baseline (Fractional Error): {fe_score.item():.4f}")
print("========================================")
print("No paper original, o FE do Baseline rondava os 0.541.")

In [ ]:
# Célula 5: PEDS Maxwell — CALIBRADO contra coarse.jl
import torch
import numpy as np
from torch.optim import Adam

from src.objects import NNstruct, CSstruct
from src.models.surrogate import PEDSModel
from src.physics.geometry import SimulationDomain, epsilon_hole_layers
from src.physics.fdfd import FDFDSolver
from src.physics.differentiable_physics import DifferentiableMaxwell
from src.models.losses import calculate_nll_loss

print("--- PEDS Maxwell (calibrado) ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Constantes do problema Maxwell (valores do notebook Julia / coarse.jl).
cs = CSstruct(resolution=10, nn_x=10, ny_nn=110,
              refsim=complex(0.3364246930443735, 0.1920021246559511))
NX, NY_NN, EPSSUB = cs.nn_x, cs.ny_nn, cs.epssub
REFSIM = complex(cs.refsim)

# 2. Um solver por frequência (ω = 2π·freq; freq = onehot·[0.5,0.75,1.0]).
def make_solver(freq):
    sd = SimulationDomain(Lx=cs.Lx, Ly=cs.Ly, omega=2*np.pi*float(freq),
                          dpml=cs.dpml, resolution=cs.resolution,
                          source=cs.source, monitor=cs.monitor)
    return FDFDSolver(sd)

_sd0 = make_solver(1.0).sd
XS, YS = _sd0.xs, _sd0.ys
NY = _sd0.ny
delta = float(XS[1] - XS[0]); Lx_grid = float(XS[-1]-XS[0]+delta)

# máscaras "magic index" do coarse.jl: ar (topo) / banda da rede / substrato (fundo)
_air  = (cs.dpml + YS) <= 0.35*(cs.Ly + 2*cs.dpml)
_band = (0.35*(cs.Ly+2*cs.dpml) < (cs.dpml+YS)) & ((cs.dpml+YS) <= 0.35*(cs.Ly+2*cs.dpml) + NY_NN/cs.resolution)
N_AIR, N_BAND, N_SUB = int(_air.sum()), int(_band.sum()), int((~_air & ~_band).sum())
assert N_BAND == NY_NN, (N_BAND, NY_NN)
print(f"grid {NY}x{NX} | bandas ar/rede/substrato = {N_AIR}/{N_BAND}/{N_SUB}")

# cache de solvers + pontes diferenciáveis por frequência
_solver_cache, _maxwell_cache = {}, {}
def get_maxwell(freq):
    k = round(float(freq), 6)
    if k not in _maxwell_cache:
        s = make_solver(k); _solver_cache[k] = s
        _maxwell_cache[k] = DifferentiableMaxwell(s)
    return _maxwell_cache[k]

# 3. Geometria coarse (prior físico) na banda da rede — NÃO diferenciável.
def build_coarse_band(X_batch):
    widths = X_batch[:, :10].detach().cpu().numpy().astype(np.float64)
    widths = np.clip(widths, delta + 1e-3, Lx_grid - 1e-3)
    bands = []
    for ps in widths:
        g = np.real(epsilon_hole_layers(XS, YS, ps,
                    refractive_indexes=tuple(cs.refracsim),
                    interstice=cs.interstice, hole=cs.hole))
        bands.append(g[_band, :])                       # get_meat -> (110,10)
    return torch.tensor(np.stack(bands, 0), dtype=torch.float32, device=X_batch.device)

# 4. buildgeom: embute a banda combinada no domínio completo (diferenciável).
_air_block = None
def buildgeom(band_eps):
    B = band_eps.shape[0]
    air = band_eps.new_ones(B, N_AIR, NX)
    sub = band_eps.new_full((B, N_SUB, NX), EPSSUB)
    return torch.cat([air, band_eps, sub], dim=1)        # (B, NY, NX)

# 5. Resolve transmissão por grupo de frequência e normaliza por refsim.
def solve_transmission(full_eps, freqs):
    B = full_eps.shape[0]
    out_r = full_eps.new_zeros(B); out_i = full_eps.new_zeros(B)
    for fval in torch.unique(freqs):
        idx = (freqs == fval).nonzero(as_tuple=True)[0]
        r, i = get_maxwell(float(fval))(full_eps[idx])   # transmissão BRUTA
        out_r = out_r.index_copy(0, idx, r)
        out_i = out_i.index_copy(0, idx, i)
    # divide por refsim complexo (ops reais p/ manter o autograd): t/refsim
    a, b = REFSIM.real, REFSIM.imag; d = a*a + b*b
    rp = (out_r*a + out_i*b) / d
    ip = (out_i*a - out_r*b) / d
    return rp, ip

# 6. Forward híbrido do PEDS.
def physics_informed_forward(X_batch, model):
    coarse_band = build_coarse_band(X_batch)             # (B,110,10), sem grad
    eps_band, vp = model(X_batch, coarse_band)           # w·NN + (1-w)·coarse, na banda
    full = buildgeom(eps_band)                           # (B,210,10)
    freqs = X_batch[:, 10:13] @ torch.tensor([0.5, 0.75, 1.0], device=X_batch.device)
    rp, ip = solve_transmission(full, freqs)             # predição = solver/refsim
    return rp, ip, vp

# 7. Modelo: saída do gerador na banda (110x10); mvar sobre a banda combinada.
nn_params = NNstruct(
    inGen=[13, 256, 256],
    outGen=[256, 256, NY_NN * NX],
    postGen=[lambda x: x * 1.5 + 2.5,
             lambda x: x.reshape(-1, NY_NN, NX)],
    inVar=[NY_NN * NX, 256, 256, 256],
    outVar=[256, 256, 256, 1],
)
peds_model = PEDSModel(nn_params).to(device)
optimizer_peds = Adam(peds_model.parameters(), lr=1e-3)

# 8. Treino (1280 amostras; ~minutos em CPU pelo custo do solver esparso).
print("--- Treino PEDS ---")
peds_model.train()
for epoch in range(10):
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_peds.zero_grad()
        rp, ip, vp_out = physics_informed_forward(X_batch, peds_model)
        vp = torch.abs(vp_out.squeeze()) + 1e-6
        loss = calculate_nll_loss((rp, ip, vp), y_batch)
        loss.backward()
        optimizer_peds.step()
        epoch_loss += loss.item()
    print(f"Epoca {epoch+1}/10 | Loss NLL: {epoch_loss/len(train_loader):.4f} | w={torch.sigmoid(peds_model.cw*peds_model.multfact).item():.3f}")

# 9. FE no teste (mesma metrica do paper; alvo PEDS ~ 0.278).
peds_model.eval()
with torch.no_grad():
    rp, ip, _ = physics_informed_forward(X_test.to(device), peds_model)
    fe = calculate_fe(rp, ip, y_test.to(device))
print(f"\\nPEDS Fractional Error (teste): {fe.item():.4f}  (paper: ~0.278)")


In [ ]:
# Célula 6: Motor de Benchmarking (PEDS vs Baseline) - TREINO REAL
from src.models.losses import calculate_nll_loss

def run_experiment(model_type='PEDS'):
    print(f"--- Iniciando Experimento: {model_type} ---")
    
    # 1. Escolher o modelo
    if model_type == 'PEDS':
        model = PEDSModel(nn_params).to(device)
    else:
        model = BaselineModel(nn_params).to(device)
        
    optimizer = Adam(model.parameters(), lr=1e-3)
    model.train()
    
    # 2. Loop de treino real
    for epoch in range(10): 
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            
            if model_type == 'PEDS':
                # Aqui o 'physics_informed_forward' fará a mágica
                rp, ip, vp_out = physics_informed_forward(X_batch, model)
            else:
                z = model.mgen(X_batch)
                pred_out = model.pred(z)
                rp, ip = pred_out[:, 0], pred_out[:, 1]
                vp_out = model.mvar(z)
                
            vp = torch.abs(vp_out.squeeze()) + 1e-6
            loss = calculate_nll_loss((rp, ip, vp), y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        print(f"Época {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")
        
    # 3. Calcular o FE final
    model.eval()
    with torch.no_grad():
        X_test_gpu, y_test_gpu = X_test.to(device), y_test.to(device)
        # Replicar a mesma lógica de forward para o teste
        if model_type == 'PEDS':
            rp, ip, _ = physics_informed_forward(X_test_gpu, model)
        else:
            z = model.mgen(X_test_gpu)
            pred_out = model.pred(z)
            rp, ip = pred_out[:, 0], pred_out[:, 1]
            
        return calculate_fe(rp, ip, y_test_gpu).item()

# Executar o Benchmark
res_peds = run_experiment('PEDS')
res_baseline = run_experiment('Baseline')

print(f"\nResultados Finais:")
print(f"PEDS FE: {res_peds:.4f}")
print(f"Baseline FE: {res_baseline:.4f}")